In [ ]:
import kagglehub

path = kagglehub.dataset_download("rupeshkumaryadav/mumbai-slum-detection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'mumbai-slum-detection-dataset' dataset.
Path to dataset files: /kaggle/input/mumbai-slum-detection-dataset


In [ ]:
import numpy as np
import os
from tqdm import tqdm

# Setup directories
save_dir_img = "data/train_images_rgb"
save_dir_mask = "data/train_masks_rgb"
os.makedirs(save_dir_img, exist_ok=True)
os.makedirs(save_dir_mask, exist_ok=True)

# Load full datasets
full_img = np.load("/kaggle/input/mumbai-slum-detection-dataset/Data/sentinel_input.npy") ]
full_mask = np.load("/kaggle/input/mumbai-slum-detection-dataset/Data/slum_labels.npy")

CHIP_SIZE = 256
STRIDE = 64
count = 0

for y in range(0, full_img.shape[1] - CHIP_SIZE, STRIDE):
    for x in range(0, full_img.shape[2] - CHIP_SIZE, STRIDE):
        img_chip = full_img[:, y:y+CHIP_SIZE, x:x+CHIP_SIZE]
        mask_chip = full_mask[y:y+CHIP_SIZE, x:x+CHIP_SIZE]

        # Only save chips that contain slums to avoid the "all-black" prediction problem
        if mask_chip.max() > 0:
            np.save(f"{save_dir_img}/chip_{count}.npy", img_chip)
            np.save(f"{save_dir_mask}/chip_{count}.npy", mask_chip)
            count += 1

print(f"✅ Created {count} RGB-capable chips.")

✅ Created 1115 RGB-capable chips.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import glob
import numpy as np # Added for np.load in Dataset

class SlumRGBDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths = img_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # Load (4, 256, 256)
        image = np.load(self.img_paths[idx]).astype('float32')
        mask = np.load(self.mask_paths[idx]).astype('float32')

        # 1. Select only first 3 bands (RGB)
        image = image[:3, :, :]

        # 2. Advanced Normalization (Per-channel)
        # We scale to 0-1 then use standard ImageNet stats
        image = image / (image.max() + 1e-8)

        # 3. Final shapes
        mask = np.expand_dims(mask, axis=0)
        return torch.from_numpy(image), torch.from_numpy(mask)

# Split and Load
img_files = sorted(glob.glob("data/train_images_rgb/*.npy"))
mask_files = sorted(glob.glob("data/train_masks_rgb/*.npy"))

from sklearn.model_selection import train_test_split


train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(img_files, mask_files, test_size=0.30, random_state=42)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(temp_imgs, temp_masks, test_size=0.50, random_state=42)

train_loader = DataLoader(SlumRGBDataset(train_imgs, train_masks), batch_size=8, shuffle=True)
val_loader = DataLoader(SlumRGBDataset(val_imgs, val_masks), batch_size=8, shuffle=False)
test_loader = DataLoader(SlumRGBDataset(test_imgs, test_masks), batch_size=8, shuffle=False)

In [ ]:
print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

Train: 780 | Val: 167 | Test: 168


In [ ]:
print(f"Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Train: 98 | Val: 21 | Test: 21


In [ ]:
pip install segmentation-models-pytorch

In [ ]:
import torch
import segmentation_models_pytorch as smp

# Initialize Model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)

In [ ]:
EPOCHS = 30
best_val_loss = float('inf')

print(f"\n Starting Training on {DEVICE}...")

for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    total_train_loss = 0
    total_train_iou = 0
    total_train_f1 = 0

    for imgs, msks in train_loader:
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)

        optimizer.zero_grad()
        output = model(imgs)
        loss = loss_fn(output, msks)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

        # Calculate metrics manually
        pred_masks = (output.sigmoid() > 0.5).float()

        # IoU (Jaccard Index)
        intersection = (pred_masks * msks).sum()
        union = (pred_masks + msks).sum() - intersection
        batch_iou = (intersection / (union + 1e-8)).item()
        total_train_iou += batch_iou

        # F1 Score (Dice Coefficient)
        tp = (pred_masks * msks).sum()
        fp = ((1 - msks) * pred_masks).sum()
        fn = ((1 - pred_masks) * msks).sum()
        batch_f1 = ((2 * tp) / (2 * tp + fp + fn + 1e-8)).item()
        total_train_f1 += batch_f1

    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_iou = total_train_iou / len(train_loader)
    avg_train_f1 = total_train_f1 / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval()
    total_val_loss = 0
    total_val_iou = 0
    total_val_f1 = 0
    with torch.no_grad():
        for imgs, msks in val_loader:
            imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
            output = model(imgs)
            val_loss = loss_fn(output, msks)
            total_val_loss += val_loss.item()

            # Calculate metrics manually
            pred_masks = (output.sigmoid() > 0.5).float()

            # IoU (Jaccard Index)
            intersection = (pred_masks * msks).sum()
            union = (pred_masks + msks).sum() - intersection
            batch_iou = (intersection / (union + 1e-8)).item()
            total_val_iou += batch_iou

            # F1 Score (Dice Coefficient)
            tp = (pred_masks * msks).sum()
            fp = ((1 - msks) * pred_masks).sum()
            fn = ((1 - pred_masks) * msks).sum()
            batch_f1 = ((2 * tp) / (2 * tp + fp + fn + 1e-8)).item()
            total_val_f1 += batch_f1

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_iou = total_val_iou / len(val_loader)
    avg_val_f1 = total_val_f1 / len(val_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Train IoU: {avg_train_iou:.4f} | Train F1: {avg_train_f1:.4f} | Val Loss: {avg_val_loss:.4f} | Val IoU: {avg_val_iou:.4f} | Val F1: {avg_val_f1:.4f}")

    # --- SAVE THE MODEL ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_slum_model_rgb.pth")
        print(f" New best model saved with Val Loss: {avg_val_loss:.4f}")

print("\n Training Complete.")



🚀 Starting Training on cuda...
Epoch [1/30] | Train Loss: 0.7467 | Train IoU: 0.2274 | Train F1: 0.3584 | Val Loss: 0.6741 | Val IoU: 0.3236 | Val F1: 0.4878
💾 New best model saved with Val Loss: 0.6741
Epoch [2/30] | Train Loss: 0.6061 | Train IoU: 0.4268 | Train F1: 0.5886 | Val Loss: 0.5382 | Val IoU: 0.5486 | Val F1: 0.7072
💾 New best model saved with Val Loss: 0.5382
Epoch [3/30] | Train Loss: 0.4807 | Train IoU: 0.5513 | Train F1: 0.7074 | Val Loss: 0.4466 | Val IoU: 0.5654 | Val F1: 0.7213
💾 New best model saved with Val Loss: 0.4466
Epoch [4/30] | Train Loss: 0.3945 | Train IoU: 0.6002 | Train F1: 0.7466 | Val Loss: 0.3519 | Val IoU: 0.6504 | Val F1: 0.7873
💾 New best model saved with Val Loss: 0.3519
Epoch [5/30] | Train Loss: 0.3249 | Train IoU: 0.6458 | Train F1: 0.7819 | Val Loss: 0.2909 | Val IoU: 0.6850 | Val F1: 0.8124
💾 New best model saved with Val Loss: 0.2909
Epoch [6/30] | Train Loss: 0.2697 | Train IoU: 0.6849 | Train F1: 0.8117 | Val Loss: 0.2525 | Val IoU: 0.699

In [ ]:
import matplotlib.pyplot as plt

model.load_state_dict(torch.load("best_slum_model_rgb.pth"))
model.eval()

all_imgs = []
all_msks = []
all_preds = []

with torch.no_grad():
    for imgs, msks in test_loader:
        preds = model(imgs.to(DEVICE)).sigmoid() > 0.25
        all_imgs.append(imgs.cpu().numpy())
        all_msks.append(msks.cpu().numpy())
        all_preds.append(preds.cpu().numpy())

imgs = np.concatenate(all_imgs)
msks = np.concatenate(all_msks)
preds = np.concatenate(all_preds)

# Display only a small number of images to avoid rendering issues
num_images_to_display = 30


if num_images_to_display > len(imgs):
    num_images_to_display = len(imgs)

fig, axes = plt.subplots(num_images_to_display, 3, figsize=(15, num_images_to_display * 5))

for i in range(num_images_to_display):
    # Convert (3, 256, 256) -> (256, 256, 3) for plotting
    rgb_img = imgs[i].transpose(1, 2, 0)
    rgb_img = (rgb_img - rgb_img.min()) / (rgb_img.max() - rgb_img.min() + 1e-8)

    # Handle cases where axes might be a single array if num_images_to_display is 1
    if num_images_to_display == 1:
        axes[0].imshow(rgb_img)
        axes[0].set_title("RGB Input")
        axes[1].imshow(msks[i][0], cmap='gray')
        axes[1].set_title("True Slum Mask")
        axes[2].imshow(preds[i][0], cmap='gray')
        axes[2].set_title("Model Prediction")
    else:
        axes[i, 0].imshow(rgb_img)
        axes[i, 0].set_title("RGB Input")
        axes[i, 1].imshow(msks[i][0], cmap='gray')
        axes[i, 1].set_title("True Slum Mask")
        axes[i, 2].imshow(preds[i][0], cmap='gray')
        axes[i, 2].set_title("Model Prediction")

plt.tight_layout()
plt.savefig("slum_segmentation_predictions.jpg")
plt.show()

In [ ]:
import torch
import segmentation_models_pytorch as smp

# Load the best model
model.load_state_dict(torch.load("best_slum_model_rgb.pth"))
model.eval()

# Define loss function (same as training)
loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)

# Initialize metrics for test set
total_test_loss = 0
total_test_iou = 0
total_test_f1 = 0

print("\n📊 Evaluating model performance on the Test Set...")

with torch.no_grad():
    for imgs, msks in test_loader:
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
        output = model(imgs)
        test_loss = loss_fn(output, msks)
        total_test_loss += test_loss.item()

        # Calculate metrics manually
        pred_masks = (output.sigmoid() > 0.5).float()

        # IoU (Jaccard Index)
        intersection = (pred_masks * msks).sum()
        union = (pred_masks + msks).sum() - intersection
        batch_iou = (intersection / (union + 1e-8)).item()
        total_test_iou += batch_iou

        # F1 Score (Dice Coefficient)
        tp = (pred_masks * msks).sum()
        fp = ((1 - msks) * pred_masks).sum()
        fn = ((1 - pred_masks) * msks).sum()
        batch_f1 = ((2 * tp) / (2 * tp + fp + fn + 1e-8)).item()
        total_test_f1 += batch_f1

avg_test_loss = total_test_loss / len(test_loader)
avg_test_iou = total_test_iou / len(test_loader)
avg_test_f1 = total_test_f1 / len(test_loader)

print(f"\n✅ Test Results: ")
print(f"   Average Loss: {avg_test_loss:.4f}")
print(f"   Average IoU: {avg_test_iou:.4f}")
print(f"   Average F1 Score: {avg_test_f1:.4f}")


📊 Evaluating model performance on the Test Set...

✅ Test Results: 
   Average Loss: 0.0909
   Average IoU: 0.8455
   Average F1 Score: 0.9160
